In [ ]:
import pickle

from matplotlib import pyplot as plt, cm
import numpy as np
import json
import os
import matplotlib.colors as mcolors

def getfolder(target_path, timepoint):
    # 构建目标路径
    file_path = os.path.join(target_path, timepoint)

    # 获取该路径下的所有文件夹
    folders = [f for f in os.listdir(file_path) if os.path.isdir(os.path.join(file_path, f))]

    # 筛选出以 'checkpoints-' 开头的文件夹，并提取其中的数字
    checkpoint_folders = []
    for folder in folders:
        if folder.startswith('checkpoint-'):
            # 提取数字部分并转换为整数
            try:
                num = int(folder.split('-')[1])
                checkpoint_folders.append((num, folder))
            except ValueError:
                continue  # 如果文件夹名后面的数字不合法，则跳过

    # 找到数字最大的一项
    if checkpoint_folders:
        max_num, max_folder = max(checkpoint_folders, key=lambda x: x[0])
        return os.path.join(target_path, timepoint,max_folder)
    else:
        print("No checkpoints found.")
def getValues(target_path,type = 'spatial'):
    # 获取历史accuracy和f1分数
    timepoints = ['0h','12h','1.5d','3d','5d','10d','WT']
    history_best = {}
    if type == 'spatial':
        for timepoint in timepoints:
            history_best[timepoint] = []
            history_item = []
            file_path = target_path+timepoint+'/history.txt'
            with open(file_path,'r') as file:
                lines = file.readlines()
            for line in lines:
                line = line.strip('\n')
                line = line.split(' ')
                history_item.append([float(line[0]),float(line[1])])
            sorted_history_item = sorted(history_item, key=lambda x: x[0],reverse=True)
            history_best[timepoint] = sorted_history_item[0]
        
    if type == 'gene':
        for timepoint in timepoints:
            history_best[timepoint] = []
            history_item = []
            file_path = getfolder(target_path,timepoint) +'/trainer_state.json'
            # 读取并解析 JSON 文件
            with open(file_path, 'r') as file:
                data = json.load(file)
            history_item = []
            for item in data['log_history']:
                if 'eval_accuracy' in item and 'eval_macro_f1' in item:
                    history_item.append([
                      item['eval_accuracy'],
                         item['eval_macro_f1']
                    ])
            sorted_history_item = sorted(history_item, key=lambda x: x[0], reverse=True)
            history_best[timepoint] = sorted_history_item[0]
    return history_best
# getValues(target_path ='./model_compare_celltype/gene_raw/',type = 'gene')
# getValues(target_path ='./model_compare_celltype/gene_aug/',type = 'gene')
# getValues(target_path ='./model_compare_celltype/spatial_raw/')
# getValues(target_path ='./model_compare_celltype/spatial_aug/')
def makeComparePlot():
    geneformer_raw_history = getValues(target_path ='../result/model_results/geneformer/',type = 'gene')
    geneformer_with_pretrained_history = getValues(target_path ='../result/model_results/gene_raw/',type = 'gene')
    geneformer_with_pretrained_history_Spatial =getValues(target_path ='../result/model_results/spatial_raw/')
    geneformer_with_pretrained_history_Spatial_Time = getValues(target_path='../result/model_results/time_spatial_raw/')
    # 提取时间点
    timepoints = ['0h', '12h', '1.5d', '3d', '5d', '10d', 'WT']
    
    
    # 处理所有模型的评分，合并为一个大列表
    score_matrix_acc = [
        [geneformer_raw_history[timepoint][0] for timepoint in timepoints],   # eval_accuracy for spatial_aug
        [geneformer_with_pretrained_history[timepoint][0] for timepoint in timepoints],  # eval_accuracy for spatial_raw
        [geneformer_with_pretrained_history_Spatial[timepoint][0] for timepoint in timepoints],  # eval_accuracy for gene_aug
        [geneformer_with_pretrained_history_Spatial_Time[timepoint][0] for timepoint in timepoints],  # eval_accuracy for gene_raw
    ]
    
    # 转换为numpy数组以便处理
    score_matrix_acc = np.array(score_matrix_acc)
    print(score_matrix_acc)
    # 如果你需要 `score_matrix_f1`，也可以像 `score_matrix_acc` 那样处理 `eval_macro_f1`
    score_matrix_f1 = [
        [geneformer_raw_history[timepoint][1] for timepoint in timepoints],   # eval_macro_f1 for spatial_aug
        [geneformer_with_pretrained_history[timepoint][1] for timepoint in timepoints],  # eval_macro_f1 for spatial_raw
        [geneformer_with_pretrained_history_Spatial[timepoint][1] for timepoint in timepoints],  # eval_macro_f1 for gene_aug
        [geneformer_with_pretrained_history_Spatial_Time[timepoint][1] for timepoint in timepoints],  # eval_macro_f1 for gene_raw
    ]
    
    # 转换为numpy数组
    score_matrix_f1 = np.array(score_matrix_f1)

    
    timepoints = ['0h', '12h', '36h', '72h', '120h', '240h', 'WT']
    # models = ['Gene-only model', 'Gene with augmentation', 'Gene with spatial', 'Gene with spatial and augmentation']
    models = ['Geneformer with planarian and ST','Geneformer with planarian and Spatial','Geneformer with planarian' ,'Geneformer']
    models = ['Geneformer','PFM','PFM with Spatial','PFM with ST']
    # 定义每列的基础颜色（时间点对应的基础颜色）
    base_colors = ['red', 'orange', 'blue', 'yellow', 'green', 'purple', 'black']
    score_matrix =  score_matrix_acc
    # 确定评分的最小值和最大值
    vmin = np.min(score_matrix)
    vmax = np.max(score_matrix)
    # vmax =0.75
    # 创建绘图
    fig, ax = plt.subplots(figsize=(8, 4))
    
    # 为每个时间点绘制不同颜色基调的圆点
    for i, model in enumerate(models):
        for j, timepoint in enumerate(timepoints):
            # 获取基础颜色
            base_color = mcolors.to_rgba(base_colors[j], alpha=1.0)  # 转换为RGBA格式，alpha为不透明度
            
            # 根据评分值调整颜色的强度
            intensity = (score_matrix[i, j] - vmin) / (vmax - vmin)  # 计算评分的强度（0到1之间）
            color = tuple([(1-intensity) * (1 - c) + c for c in base_color[:3]])  # 混合基础颜色和评分强度，保持alpha值不变
            
            # 在对应位置绘制圆形点
            ax.scatter(j, i*1, color=color, s=2000, edgecolor='black', marker='o', zorder=10)
          
    
    # 设置x和y轴标签
    ax.set_xticks(np.arange(len(timepoints)))
    ax.set_xticklabels(timepoints, rotation=45)
    ax.set_yticks(np.linspace(0, len(models) - 1, len(models)) * 1)  # 让 y 轴点间距变小
    ax.set_yticklabels(models, rotation=0)
    ax.set_ylim(-0.9, len(models))  # 增加上下留白
    ax.set_xlim(-0.9, len(timepoints))  # 增加上下留白
    # 添加颜色条
    # 这里使用了颜色的渐变来表示不同的评分强度
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap='Greys', norm=plt.Normalize(vmin=vmin, vmax=vmax)), ax=ax)
    # cbar.set_label('Model Score')
    
    # 添加标题
    ax.set_title('Model Acc-Score Performance Over Time')
    
    # 调整布局
    plt.tight_layout(pad = 20.0)
    
    plt.savefig("../result/model_performance_acc.pdf", dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig("../result/model_performance_acc.svg", bbox_inches='tight', format='svg')
    score_matrix =  score_matrix_f1
    # 确定评分的最小值和最大值
    vmin = np.min(score_matrix)
    vmax = np.max(score_matrix)
    # vmax =0.75
    # 创建绘图
    fig, ax = plt.subplots(figsize=(8, 4))
    
    # 为每个时间点绘制不同颜色基调的圆点
    for i, model in enumerate(models):
        for j, timepoint in enumerate(timepoints):
            # 获取基础颜色
            base_color = mcolors.to_rgba(base_colors[j], alpha=1.0)  # 转换为RGBA格式，alpha为不透明度
            
            # 根据评分值调整颜色的强度
            intensity = (score_matrix[i, j] - vmin) / (vmax - vmin)  # 计算评分的强度（0到1之间）
            color = tuple([(1-intensity) * (1 - c) + c for c in base_color[:3]])  # 混合基础颜色和评分强度，保持alpha值不变
            
            # 在对应位置绘制圆形点
            ax.scatter(j, i*1, color=color, s=2000, edgecolor='black', marker='o', zorder=10)
          
    
    # 设置x和y轴标签
    ax.set_xticks(np.arange(len(timepoints)))
    ax.set_xticklabels(timepoints, rotation=45)
    ax.set_yticks(np.linspace(0, len(models) - 1, len(models)) * 1)  # 让 y 轴点间距变小
    ax.set_yticklabels(models, rotation=0)
    ax.set_ylim(-0.9, len(models))  # 增加上下留白
    ax.set_xlim(-0.9, len(timepoints))  # 增加上下留白
    # 添加颜色条
    # 这里使用了颜色的渐变来表示不同的评分强度
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap='Greys', norm=plt.Normalize(vmin=vmin, vmax=vmax)), ax=ax)
    # cbar.set_label('Model Score')
    
    # 添加标题
    ax.set_title('Model F1-Score Performance Over Time')
    
    # 调整布局
    plt.tight_layout(pad = 20.0)
    
    plt.savefig("../result/model_performance_f1.pdf", dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig("../result/model_performance_f1.svg", bbox_inches='tight', format='svg')
    
    
    # plt.savefig('Acc.png',dpi = 3000)

In [ ]:
makeComparePlot()

In [ ]:
import pickle
def makeComparePlot2():
    # 不同模型的结果比较，使用条形图展示
    # 读取scGPT的结果
    timepoints = ['0h', '12h', '1.5d', '3d', '5d', '10d', 'WT']
    timepoints_name = ['0h', '12h', '36h', '72h', '120h', '240h', 'WT']
    history_scgpt_acc = []
    history_scgpt_f1 = []
    for timepoint in timepoints:
        with open('../scGPT/scGPT-main/save/dev_'+timepoint+'/model_history.pkl', 'rb') as f:
            history_timepoint = pickle.load(f)
            #print(history_timepoint)
            sorted_history_item = sorted(history_timepoint, key=lambda x: x['test/accuracy'], reverse=True)
            sorted_history_item = sorted_history_item[0]
            history_scgpt_acc.append(sorted_history_item['test/accuracy'])
            history_scgpt_f1.append(sorted_history_item['test/macro_f1'])
    history_scBERT = {}
    for timepoint in timepoints:
        with open('../scBERT/scBERT-master/ckpts/'+timepoint+'/history.txt', 'r') as f:
            lines = f.readlines()
            history_item = []
            for line in lines:
                line = line.strip('\n')
                line = line.split(' ')
                history_item.append([float(line[0]),float(line[1])])
            sorted_history_item = sorted(history_item, key=lambda x: x[0],reverse=True)
            history_scBERT[timepoint] = sorted_history_item[0]
        # history_scBERT['WT'] = [0.33,0.05]
    gene_raw_history = getValues(target_path ='../result/model_results/gene_raw/',type = 'gene')
    geneformer_history = getValues(target_path ='../result/model_results/geneformer/',type = 'gene')
    spatial_time_history = getValues(target_path ='../result/model_results/time_spatial_raw/',type = 'spatial')
    geneformer_acc = [geneformer_history[timepoint][0] for timepoint in timepoints]
    geneformer_pretrained_acc = [gene_raw_history[timepoint][0] for timepoint in timepoints]
    scBERT_acc = [history_scBERT[timepoint][0] for timepoint in timepoints]
    geneformer_ST_acc = [ spatial_time_history[timepoint][0] for timepoint in timepoints]
    geneformer_f1 = [geneformer_history[timepoint][1] for timepoint in timepoints]
    geneformer_pretrained_f1 = [gene_raw_history[timepoint][1] for timepoint in timepoints]
    scBERT_f1 = [history_scBERT[timepoint][1] for timepoint in timepoints]
    geneformer_ST_f1 = [spatial_time_history[timepoint][1] for timepoint in timepoints]
    scgpt_acc = history_scgpt_acc
    scgpt_f1 = history_scgpt_f1
    
    # 设置条形宽度
    bar_width = 0.18
    x = np.arange(len(timepoints))
    
    # 绘制 Accuracy 对比
    plt.figure(figsize=(12, 5))
    plt.bar(x - bar_width*2,scBERT_acc, width=bar_width, label="scBERT Acc", color='b', alpha=0.7)
    plt.bar(x - bar_width, geneformer_acc, width=bar_width, label="Geneformer Acc", color='y', alpha=0.7)
    plt.bar(x , scgpt_acc, width=bar_width, label="scGPT Acc", color='r', alpha=0.7)
    plt.bar(x+bar_width, geneformer_pretrained_acc, width=bar_width, label="PFM Acc", color='brown', alpha=0.7)
    plt.bar(x+bar_width*2, geneformer_ST_acc, width=bar_width, label="PFM with ST Acc", color='g', alpha=0.7)
 
    plt.ylim(0.25, 0.9)  # 限定 y 轴范围
    plt.xticks(x, timepoints_name)
    plt.ylabel("Accuracy")
    plt.title("Model Accuracy Comparison")
    plt.legend()
    # plt.show()
    plt.savefig("model_compare_acc.pdf", dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig("model_compare_acc.svg", bbox_inches='tight', format='svg')
    
    # 绘制 F1-score 对比
    plt.figure(figsize=(12, 5))
    plt.bar(x- bar_width*2,  scBERT_f1, width=bar_width, label="scBERT F1", color='b', alpha=0.7)
    plt.bar(x- bar_width,geneformer_f1, width=bar_width, label="Geneformer F1", color='y', alpha=0.7)
    plt.bar(x , scgpt_f1, width=bar_width, label="scGPT F1", color='r', alpha=0.7)
    plt.bar(x +  bar_width,  geneformer_pretrained_f1, width=bar_width, label="PFM F1", color='brown', alpha=0.7)
    plt.bar(x +  bar_width*2, geneformer_ST_f1, width=bar_width, label="PFM with ST F1", color='g', alpha=0.7)
    plt.ylim(0.0, 0.75)  # 限定 y 轴范围
    plt.xticks(x, timepoints_name)
    plt.ylabel("F1-score")
    plt.title("Model F1-score Comparison")
    plt.legend()
    plt.savefig("model_compare_f1.pdf", dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig("model_compare_f1.svg", bbox_inches='tight', format='svg')
    
makeComparePlot2()